In [1]:
import torch
import torch.nn as nn
import math
from pathlib import Path

In [2]:
# =========================================================
# 1. DATA
# =========================================================

text = "The cat sat on the mat because it was tired. After the cat rested for a while, the animal walked to the window and watched the birds outside. The birds flew over the garden, while the tired cat quietly followed them with its eyes."

chars = sorted(set(text))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

print("stoi:", stoi)
print("itos:", itos)


data = torch.tensor(
    [stoi[ch] for ch in text],
    dtype=torch.long
)



stoi: {' ': 0, ',': 1, '.': 2, 'A': 3, 'T': 4, 'a': 5, 'b': 6, 'c': 7, 'd': 8, 'e': 9, 'f': 10, 'g': 11, 'h': 12, 'i': 13, 'k': 14, 'l': 15, 'm': 16, 'n': 17, 'o': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'y': 26}
itos: {0: ' ', 1: ',', 2: '.', 3: 'A', 4: 'T', 5: 'a', 6: 'b', 7: 'c', 8: 'd', 9: 'e', 10: 'f', 11: 'g', 12: 'h', 13: 'i', 14: 'k', 15: 'l', 16: 'm', 17: 'n', 18: 'o', 19: 'q', 20: 'r', 21: 's', 22: 't', 23: 'u', 24: 'v', 25: 'w', 26: 'y'}


In [3]:
# =========================================================
# 2. TRAINING DATA
# =========================================================

D_MODEL = 128
NUM_HEADS = 4
HEAD_DIM = D_MODEL // NUM_HEADS
BLOCK_SIZE = 6
VOCAB_SIZE = len(chars)


data = torch.tensor([stoi[x] for x in text])
print(data.size())


index = 0
x_str = "".join(itos[c.item()] for c in data[index : index + BLOCK_SIZE])
y_str = "".join(itos[c.item()] for c in data[index + 1 : index + BLOCK_SIZE + 1])

print("X:", repr(x_str), "-> Y:", repr(y_str))


X =  []
Y =  []

for i in range(len(data) - BLOCK_SIZE):
    X.append( data[i: i + BLOCK_SIZE] )
    Y.append(data[i + 1 : i + BLOCK_SIZE + 1])

X = torch.stack(X)
Y = torch.stack(Y)


print("vocab_size:", VOCAB_SIZE)
print("X shape:", X.shape)
print("Y shape:", Y.shape)




torch.Size([230])
X: 'The ca' -> Y: 'he cat'
vocab_size: 27
X shape: torch.Size([224, 6])
Y shape: torch.Size([224, 6])


In [4]:
# =========================================================
# 3. POSITIONAL ENCODING
# =========================================================

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        # Even dimensions
        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # [1, max_len, d_model]
        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        # x:
        # [batch, sequence, d_model]

        seq_len = x.shape[1]

        return x + self.pe[:, :seq_len]

In [5]:
# =========================================================
# 4. MULTIHEAD-ATTENTION
# =========================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0


        self.d_model = d_model
        self.num_heads  = num_heads
        self.head_dim = d_model // num_heads

   
        self.Wq = nn.Linear(
            d_model,
            d_model
        )

        self.Wk = nn.Linear(
            d_model,
            d_model
        )

        self.Wv = nn.Linear(
            d_model,
            d_model
        )

        self.Wo = nn.Linear(
            d_model,
            d_model
        )

    def forward(self, x):
   
        B, T, C = x.shape # [226, 4, 32]

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        
        


        Q = Q.view(B,T,self.num_heads,self.head_dim) # [226, 4, 2, 16]
        K = K.view(B,T,self.num_heads,self.head_dim)
        V = V.view(B,T,self.num_heads,self.head_dim)


        
        Q = Q.transpose(1,2) # [226, 2, 4, 16]
        K = K.transpose(1,2)
        V = V.transpose(1,2)


        KT = K.transpose(-2, -1)
        
        scores = Q @ KT #[226, 2, 4, 4]
        scores = scores / math.sqrt(
            self.head_dim
        ) 




        # -----------------------------------------
        # Causal mask
        # -----------------------------------------

        mask = torch.tril(
            torch.ones(T, T)
        )


        
        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )

        weights = torch.softmax(
            scores,
            dim=-1
        )

        out = weights @ V

        out = out.transpose(1, 2)

        out = out.contiguous().view(
            B,
            T,
            self.d_model
        )


        out = self.Wo(out)

        return out





In [6]:
# -------------------------
# Transformer Block
# -------------------------

class TransformerBlock(nn.Module):

    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attention = MultiHeadAttention(D_MODEL, NUM_HEADS)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.fc1 = nn.Linear(D_MODEL, 4 * D_MODEL)
        self.fc2 = nn.Linear(4 * D_MODEL, D_MODEL)

    def forward(self, x):

        x_norm1 = self.ln1(x)
        attention_output = self.attention(self.ln1(x))
        x = x + attention_output

        mlp_output = self.fc1(self.ln2(x))
        mlp_output = torch.relu(mlp_output)
        mlp_output = self.fc2(mlp_output)
        x = x + mlp_output


        return x

In [7]:
class Model(nn.Module):

    def __init__(self):

        super().__init__()

        # Character embedding
        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL
        )

        # ONLY ONE Transformer Block
        self.block = TransformerBlock()

   
        # Convert hidden state → character logits
        self.fc = nn.Linear(
            D_MODEL,
            VOCAB_SIZE
        )

    def forward(self, x):

        # [B,T]
        x = self.embedding(x)

        # [B,T,D_MODEL]
        x = self.block(x)


        # [B,T,VOCAB_SIZE]
        logits = self.fc(x)

        return logits

In [8]:

# =========================================================
# 6. TRAINING
# =========================================================

model = Model()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.01
)

for step in range(1000):

    # [B, T, 27]
    logits = model(X)


    # logits -> [N, C] target -> [N]
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        Y.reshape(-1)
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 200 == 0:

        print(
            "step:",
            step,
            "loss:",
            loss.item()
        )

step: 0 loss: 3.4011168479919434
step: 200 loss: 0.47472909092903137
step: 400 loss: 0.4607965350151062
step: 600 loss: 0.45744818449020386
step: 800 loss: 0.49565255641937256


In [9]:
# =========================================================
# 7. GENERATION
# =========================================================

model.eval()

context = torch.tensor([
    [stoi["a"],
     stoi["n"],
     stoi["i"]]
])

result = "ani"

with torch.no_grad():

    for _ in range(30):

        # -----------------------------------------
        # Model prediction
        # -----------------------------------------

        logits = model(context)

        # Last position
        logits = logits[:, -1, :]

        # -----------------------------------------
        # Convert logits → probabilities
        # -----------------------------------------

        probs = torch.softmax(
            logits,
            dim=-1
        )

        # -----------------------------------------
        # Sample
        # -----------------------------------------

        next_char = torch.multinomial(
            probs,
            num_samples=1
        )

        # -----------------------------------------
        # ID → character
        # -----------------------------------------

        result += itos[
            next_char.item()
        ]

        # -----------------------------------------
        # Add token to context
        # -----------------------------------------

        # Append new character, then keep only the
        # last BLOCK_SIZE tokens (positional encoding
        # is only built for max_len = BLOCK_SIZE)
        context = torch.cat(
            [context, next_char],
            dim=1
        )[:, -BLOCK_SIZE:]


print("\nGenerated:")
print(result)



Generated:
animal walked to the with inemal 
